# UQ Evaluation Metrics

Comprehensive evaluation of uncertainty quality.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from deepuq.metrics import (
    prediction_interval_coverage,
    negative_log_likelihood,
    continuous_ranked_probability_score,
    interval_score,
    auroc_ood,
    fpr_at_tpr,
    aurc,
    risk_coverage_curve,
)
from deepuq.models import MLP
from deepuq.methods.mc_dropout import MCDropoutWrapper

## Setup: Train Model with UQ

In [ ]:
# Generate data
np.random.seed(42)
torch.manual_seed(42)

X = np.linspace(-3, 3, 200).reshape(-1, 1)
y = np.sin(X) + 0.1 * X**2 + np.random.randn(*X.shape) * 0.1

X_train = torch.tensor(X, dtype=torch.float32)
y_train = torch.tensor(y, dtype=torch.float32)

# Train a model with dropout for MC Dropout UQ
model = MLP(1, [64, 64], 1, p_drop=0.1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
for epoch in range(500):
    optimizer.zero_grad()
    pred = model(X_train)
    loss = torch.nn.functional.mse_loss(pred, y_train)
    loss.backward()
    optimizer.step()

ensemble = MCDropoutWrapper(model, n_mc=20, apply_softmax=False)

# Get predictions
X_test = torch.tensor(np.linspace(-3, 3, 100).reshape(-1, 1), dtype=torch.float32)
y_test = torch.tensor(np.sin(X_test.numpy()) + 0.1 * X_test.numpy()**2 + np.random.randn(100, 1) * 0.1, dtype=torch.float32)

result = ensemble.predict_uq(X_test)
mean = result.mean.squeeze()
std = result.total_var.sqrt().squeeze()
print(f"Predictions: mean shape={mean.shape}, std shape={std.shape}")

## Calibration Metrics

In [ ]:
# Compute PICP at different confidence levels
from scipy.stats import norm

confidence_levels = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99]
picp_values = []

for alpha in confidence_levels:
    z = norm.ppf((1 + alpha) / 2)
    lower = mean - z * std
    upper = mean + z * std
    picp = prediction_interval_coverage(lower.numpy(), upper.numpy(), y_test.squeeze().numpy())
    picp_values.append(picp)

# Plot calibration curve
plt.figure(figsize=(6, 6))
plt.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
plt.plot(confidence_levels, picp_values, "bo-", label="Model")
plt.xlabel("Expected coverage")
plt.ylabel("Observed coverage (PICP)")
plt.title("Calibration Curve")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Calibration results:")
for alpha, picp in zip(confidence_levels, picp_values):
    print(f"  {alpha*100:.0f}% interval: PICP = {picp:.3f}")

## Scoring Rules

In [ ]:
# Compute scoring rules
nll = negative_log_likelihood(mean.numpy(), (std**2).numpy(), y_test.squeeze().numpy())
crps = continuous_ranked_probability_score(mean.numpy(), std.numpy(), y_test.squeeze().numpy())

z90 = norm.ppf(0.95)
lower_90 = mean - z90 * std
upper_90 = mean + z90 * std
is_90 = interval_score(lower_90.numpy(), upper_90.numpy(), y_test.squeeze().numpy(), alpha=0.1)

print("Scoring Rules:")
print(f"{'Metric':<20} {'Value':<10}")
print("-" * 30)
print(f"{'NLL':<20} {nll:.4f}")
print(f"{'CRPS':<20} {crps:.4f}")
print(f"{'Interval Score (90%)':<20} {is_90:.4f}")

## OOD Detection

In [ ]:
# In-distribution data
X_id = torch.tensor(np.random.uniform(-3, 3, (100, 1)), dtype=torch.float32)
result_id = ensemble.predict_uq(X_id)
std_id = result_id.total_var.sqrt().squeeze()

# OOD data (outside training range)
X_ood = torch.tensor(np.random.uniform(5, 8, (100, 1)), dtype=torch.float32)
result_ood = ensemble.predict_uq(X_ood)
std_ood = result_ood.total_var.sqrt().squeeze()

# Use uncertainty as OOD score
scores_id = std_id.numpy().ravel()
scores_ood = std_ood.numpy().ravel()

# Compute OOD detection metrics
auroc_val = auroc_ood(scores_id, scores_ood)
fpr95 = fpr_at_tpr(scores_id, scores_ood, tpr=0.95)

print(f"OOD Detection Results:")
print(f"  AUROC: {auroc_val:.4f}")
print(f"  FPR@95% TPR: {fpr95:.4f}")

# Visualize score distributions
plt.figure(figsize=(8, 4))
plt.hist(scores_id, bins=30, alpha=0.5, label="In-distribution", density=True)
plt.hist(scores_ood, bins=30, alpha=0.5, label="OOD", density=True)
plt.xlabel("Uncertainty (std)")
plt.ylabel("Density")
plt.title("Uncertainty Scores: ID vs OOD")
plt.legend()
plt.tight_layout()
plt.show()

## Selective Prediction Metrics

In [ ]:
# Compute AURC and plot risk-coverage curve
errors = ((mean - y_test.squeeze())**2).numpy()
uncertainties = std.numpy()
aurc_val = aurc(uncertainties, errors)
coverages, risks = risk_coverage_curve(uncertainties, errors)

print(f"AURC: {aurc_val:.4f}")

plt.figure(figsize=(8, 5))
plt.plot(coverages, risks, "b-", linewidth=2)
plt.xlabel("Coverage")
plt.ylabel("Risk (MSE)")
plt.title("Risk-Coverage Curve")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nSelective prediction: rejecting uncertain predictions reduces risk")
print(f"  Risk at 100% coverage: {risks[-1]:.4f}")
print(f"  Risk at 80% coverage: {risks[int(0.8*len(risks))]:.4f}")
print(f"  Risk at 50% coverage: {risks[int(0.5*len(risks))]:.4f}")